# 03 — Multi-Target Prediction from hBehaveMAE Embeddings

Predict **strain**, **age** (Adult vs Old), and **experimental stage** (HAB, ACQ, …)
directly from the raw 192D embeddings — bypassing clustering entirely.

### Recording structure
Each mouse has **6 recordings**: 3 experimental stages × 2 ages.
Video: `HDP-013893_ACQ_Old_4` → `animal_id / exp_stage / age / session`.

### Methodological guardrails
1. **Independence:** 3000-frame chunks, 1800-frame gaps.
2. **Mean pooling:** Each chunk → single 192D vector.
3. **Grouped CV:** `GroupShuffleSplit` on `animal_id` for strain prediction
   (forces learning of strain phenotype, not individual mouse identity).
4. For **age** and **exp_stage**, we also test grouped splits,
   but since every mouse appears at both ages / all stages,
   we additionally show stratified CV to measure within-animal discriminability.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (
    GroupShuffleSplit, StratifiedKFold,
    cross_val_predict, cross_val_score,
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    balanced_accuracy_score, accuracy_score,
)
import warnings
warnings.filterwarnings("ignore")

sns.set_context("notebook", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
print("Libraries loaded.")

## 1. Configuration

In [ ]:
from pathlib import Path

def find_base_dir(anchor="data"):
    """Walk upward from CWD until we find a child directory named `anchor`."""
    cwd = Path.cwd()
    for parent in [cwd] + list(cwd.parents):
        if (parent / anchor).is_dir():
            return parent
    raise FileNotFoundError(
        f"Could not find a '{anchor}/' directory in any parent of {cwd}."
    )

BASE_DIR = find_base_dir("data")
PROJECT_ROOT = Path("/scratch/michal/projects/dvc_ofd_2025")
EMB_H5_PATH = BASE_DIR / "data/embeddings/ofd_tailhip_20260226-205043.h5"
META_TSV_PATH = PROJECT_ROOT / "data/raw/openfield_ORT/hdp_meta.tsv"
# --- Independence parameters ---
# hBehaveMAE receptive field = 900 frames.
# 3000-frame chunks with 1800-frame gaps → zero information leakage.
STAGE = "stage2"            # 192D
CHUNK_FRAMES = 3000
GAP_FRAMES = 1800
STRIDE = CHUNK_FRAMES + GAP_FRAMES
# --- Filtering ---
HYBRID_STRAIN = "B6CAST-129SPWK-F2"
MIN_CHUNKS_PER_STRAIN = 20

POOLING = "mean"
N_SPLITS = 5
TEST_SIZE = 0.20
RANDOM_STATE = 42

print(f"BASE_DIR: {BASE_DIR}")
print(f"Embedding: {EMB_H5_PATH}  (exists: {EMB_H5_PATH.exists()})")

## 2. Load Metadata

In [ ]:
meta_df = pd.read_csv(META_TSV_PATH, sep=r'\s+')
meta_df.columns = meta_df.columns.str.replace('"', '').str.strip()
meta_df['animal_id'] = meta_df['animal_id'].astype(str).str.replace('"', '').str.strip()

strain_lookup = dict(zip(meta_df['animal_id'], meta_df['strain']))
print(f"Metadata loaded: {len(meta_df)} animals, {meta_df['strain'].nunique()} unique strains")

## 3. Extract & Pool Independent Chunks

In [ ]:
def extract_independent_chunks(embeddings, chunk_frames=3000, gap_frames=1800):
    """Slice (T, D) into independent chunks separated by gaps.

    If the video is shorter than chunk_frames, the entire video is
    returned as a single chunk (graceful fallback so no video is
    silently dropped when chunk_frames is set very large).
    """
    T = embeddings.shape[0]

    # Fallback: video shorter than one chunk → use the whole video
    if T < chunk_frames:
        return [embeddings]

    stride = chunk_frames + gap_frames
    chunks = []
    start = 0
    while start + chunk_frames <= T:
        chunks.append(embeddings[start : start + chunk_frames])
        start += stride
    return chunks
all_chunks = []
chunk_meta_rows = []

with h5py.File(EMB_H5_PATH, 'r') as f:
    video_names = sorted(f.keys())
    print(f"Videos in H5 file: {len(video_names)}")

    for vid in video_names:
        if STAGE not in f[vid]:
            continue

        emb = f[vid][STAGE][:]         # (T, 192)

        # Parse: HDP-013893_ACQ_Old_4 → animal_id, exp_stage, age
        parts = vid.split('_')
        if len(parts) < 4:
            continue
        animal_id = parts[0]
        exp_stage = parts[1]           # HAB, ACQ, etc.
        age       = parts[2]           # Adult, Old

        chunks = extract_independent_chunks(emb, CHUNK_FRAMES, GAP_FRAMES)

        for ci, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            chunk_meta_rows.append({
                "pos": len(all_chunks) - 1,
                "video_name": vid,
                "animal_id": animal_id,
                "exp_stage": exp_stage,
                "age": age,
                "chunk_idx": ci,
            })

chunk_df = pd.DataFrame(chunk_meta_rows)
print(f"\nExtracted {len(all_chunks)} independent chunks")
print(f"  Videos:  {chunk_df['video_name'].nunique()}")
print(f"  Animals: {chunk_df['animal_id'].nunique()}")
print(f"  Exp stages: {sorted(chunk_df['exp_stage'].unique())}")
print(f"  Ages:       {sorted(chunk_df['age'].unique())}")
print(f"\nRecordings per animal (videos): "
      f"{chunk_df.groupby('animal_id')['video_name'].nunique().describe()[['mean','min','max']].to_dict()}")

## 4. Data Filtering & Mean Pooling

In [ ]:
print("=" * 60)
print("Applying Strict Data Filtering")
print("=" * 60)

# Step 1: Merge with metadata
master_df = pd.merge(chunk_df, meta_df, on='animal_id', how='inner')
print(f"After metadata merge: {len(master_df)} chunks")

if master_df.empty:
    raise RuntimeError(
        f"Merge produced 0 rows! Check animal_id formats.\n"
        f"  chunk IDs: {chunk_df['animal_id'].unique()[:3]}\n"
        f"  meta IDs:  {meta_df['animal_id'].unique()[:3]}"
    )

# Step 2: Drop missing strain
master_df = master_df.dropna(subset=['strain'])
print(f"After dropping missing strain: {len(master_df)}")

# Step 3: Remove hybrid strain
n_hybrid = (master_df['strain'] == HYBRID_STRAIN).sum()
master_df = master_df[master_df['strain'] != HYBRID_STRAIN]
print(f"Removed {n_hybrid} chunks from '{HYBRID_STRAIN}': {len(master_df)} remain")

# Step 4: Minimum chunk count per strain
strain_counts = master_df['strain'].value_counts()
valid_strains = strain_counts[strain_counts >= MIN_CHUNKS_PER_STRAIN].index
n_dropped = master_df['strain'].nunique() - len(valid_strains)
master_df = master_df[master_df['strain'].isin(valid_strains)]
print(f"Removed {n_dropped} rare strains (< {MIN_CHUNKS_PER_STRAIN} chunks): {len(master_df)} remain")

# Step 5: Sync numpy arrays
filtered_chunks = [all_chunks[i] for i in master_df['pos'].values]
master_df = master_df.reset_index(drop=True)

print(f"\nFinal dataset: {len(master_df)} chunks, "
      f"{master_df['strain'].nunique()} strains, "
      f"{master_df['animal_id'].nunique()} animals")
print(f"\nBreakdown by exp_stage × age:")
print(master_df.groupby(['age', 'exp_stage']).size().unstack(fill_value=0))


# Mean-pool
X = np.array([c.mean(axis=0) for c in filtered_chunks])
print(f"\nFeature matrix: {X.shape}")
print(f"\nTarget distributions:")
for col in ['strain', 'age', 'exp_stage']:
    print(f"  {col}: {master_df[col].nunique()} unique — {master_df[col].value_counts().head(5).to_dict()}")

## 5. Verify Group Integrity

For strain prediction, `GroupShuffleSplit` on `animal_id` ensures no mouse leaks
between train and test.

In [ ]:
groups = master_df['animal_id'].values

gss = GroupShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_STATE)

print(f"Verifying animal-level separation ({N_SPLITS} splits):")
y_strain = master_df['strain'].values
for i, (tr, te) in enumerate(gss.split(X, y_strain, groups)):
    overlap = set(groups[tr]) & set(groups[te])
    print(f"  Split {i}: train={len(set(groups[tr]))} | test={len(set(groups[te]))} animals | overlap={len(overlap)}")
    assert len(overlap) == 0
print("✅ Zero leakage.")

---
# Part A: Strain Prediction (Grouped by Animal)

This is the hardest and most important test.  The model must learn
strain-level behavioral phenotypes, not individual mouse signatures.

In [ ]:
le_strain = LabelEncoder()
y_strain_enc = le_strain.fit_transform(master_df['strain'].values)
strain_names = le_strain.classes_
n_strains = len(strain_names)

models = {
    "LogReg (L2)": make_pipeline(StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0,
                           multi_class="multinomial", random_state=RANDOM_STATE)),
    "LogReg (L1)": make_pipeline(StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0,
                           penalty="l1", solver="saga", multi_class="multinomial",
                           random_state=RANDOM_STATE)),
    "Linear SVM": make_pipeline(StandardScaler(),
        LinearSVC(max_iter=5000, class_weight="balanced", C=1.0, random_state=RANDOM_STATE)),
}

gss = GroupShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_STATE)
strain_cv = {}

print("=" * 60)
print("  STRAIN PREDICTION  (GroupShuffleSplit on animal_id)")
print("=" * 60)
for name, model in models.items():
    scores = cross_val_score(model, X, y_strain_enc, groups=groups,
                              cv=gss, scoring="balanced_accuracy")
    strain_cv[name] = scores
    print(f"  {name:20s}  {scores.mean():.3f} ± {scores.std():.3f}")
print(f"\n  Chance: {1/n_strains:.3f}")

## 6. Strain — Detailed Report & Confusion Matrix

In [ ]:
best_strain = max(strain_cv, key=lambda m: strain_cv[m].mean())
gss_p = GroupShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_STATE)
y_pred_strain = cross_val_predict(models[best_strain], X, y_strain_enc, groups=groups, cv=gss_p)

print(f"Best: {best_strain}\n")
print(classification_report(y_strain_enc, y_pred_strain, target_names=strain_names, digits=3, zero_division=0))

cm = confusion_matrix(y_strain_enc, y_pred_strain, normalize='true')
fig, ax = plt.subplots(figsize=(max(10, n_strains*0.5), max(8, n_strains*0.4)))
sns.heatmap(cm, xticklabels=strain_names, yticklabels=strain_names,
            cmap='magma', vmin=0, vmax=1, linewidths=0.3, ax=ax,
            cbar_kws={'label': 'Recall'})
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Strain Confusion Matrix — {best_strain}\n(GroupShuffleSplit)")
plt.xticks(rotation=90, fontsize=6); plt.yticks(fontsize=6)
plt.tight_layout(); plt.show()

## 7. Per-Strain Recall

In [ ]:
per_strain = pd.DataFrame({
    "strain": strain_names, "recall": np.diag(cm),
    "n_chunks": [(y_strain_enc == i).sum() for i in range(n_strains)],
    "n_animals": [len(np.unique(groups[y_strain_enc == i])) for i in range(n_strains)],
}).sort_values("recall", ascending=True)

fig, ax = plt.subplots(figsize=(10, max(5, n_strains*0.25)))
ax.barh(per_strain['strain'], per_strain['recall'], color=plt.cm.RdYlGn(per_strain['recall'].values))
ax.axvline(1/n_strains, color='red', ls='--', alpha=0.5, label=f'Chance ({1/n_strains:.2f})')
ax.set_xlabel("Recall"); ax.set_title("Per-Strain Accuracy"); ax.legend(); ax.set_xlim(0,1)
plt.tight_layout(); plt.show()

---
# Part B: Age Prediction (Adult vs Old)

Binary classification. Since every mouse appears at both ages, GroupShuffleSplit
on `animal_id` tests whether the model generalizes the *aging signature*
to unseen mice (not just memorizing individuals).

In [ ]:
le_age = LabelEncoder()
y_age_enc = le_age.fit_transform(master_df['age'].values)
age_names = le_age.classes_
n_ages = len(age_names)

gss_age = GroupShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_STATE)

print("=" * 60)
print("  AGE PREDICTION  (Adult vs Old)")
print("=" * 60)

age_cv = {}
for name, model in models.items():
    # Grouped by animal — tests generalization to unseen mice
    scores_grp = cross_val_score(model, X, y_age_enc, groups=groups,
                                  cv=gss_age, scoring="balanced_accuracy")
    # Stratified — tests within-animal age discriminability
    scores_strat = cross_val_score(model, X, y_age_enc,
                                    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                                    scoring="balanced_accuracy")
    age_cv[name] = {"grouped": scores_grp, "stratified": scores_strat}
    print(f"  {name:20s}  Grouped: {scores_grp.mean():.3f}±{scores_grp.std():.3f}"
          f"  |  Stratified: {scores_strat.mean():.3f}±{scores_strat.std():.3f}")
print(f"\n  Chance: {1/n_ages:.3f}")

In [ ]:
# Age confusion matrix (best grouped model)
best_age = max(age_cv, key=lambda m: age_cv[m]["grouped"].mean())
gss_ap = GroupShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_STATE)
y_pred_age = cross_val_predict(models[best_age], X, y_age_enc, groups=groups, cv=gss_ap)

print(f"Best (grouped): {best_age}\n")
print(classification_report(y_age_enc, y_pred_age, target_names=age_names, digits=3))

cm_age = confusion_matrix(y_age_enc, y_pred_age, normalize='true')
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_age, xticklabels=age_names, yticklabels=age_names,
            annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Age Confusion Matrix — {best_age}"); plt.tight_layout(); plt.show()

---
# Part C: Experimental Stage Prediction (HAB, ACQ, …)

Multi-class. Every mouse has recordings at all stages, so GroupShuffleSplit
tests generalization to unseen mice; StratifiedKFold measures raw discriminability.

In [ ]:
le_exp = LabelEncoder()
y_exp_enc = le_exp.fit_transform(master_df['exp_stage'].values)
exp_names = le_exp.classes_
n_exp = len(exp_names)

gss_exp = GroupShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_STATE)

print("=" * 60)
print("  EXPERIMENTAL STAGE PREDICTION")
print("=" * 60)

exp_cv = {}
for name, model in models.items():
    scores_grp = cross_val_score(model, X, y_exp_enc, groups=groups,
                                  cv=gss_exp, scoring="balanced_accuracy")
    scores_strat = cross_val_score(model, X, y_exp_enc,
                                    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                                    scoring="balanced_accuracy")
    exp_cv[name] = {"grouped": scores_grp, "stratified": scores_strat}
    print(f"  {name:20s}  Grouped: {scores_grp.mean():.3f}±{scores_grp.std():.3f}"
          f"  |  Stratified: {scores_strat.mean():.3f}±{scores_strat.std():.3f}")
print(f"\n  Chance: {1/n_exp:.3f}")

In [ ]:
best_exp = max(exp_cv, key=lambda m: exp_cv[m]["grouped"].mean())
gss_ep = GroupShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_STATE)
y_pred_exp = cross_val_predict(models[best_exp], X, y_exp_enc, groups=groups, cv=gss_ep)

print(f"Best (grouped): {best_exp}\n")
print(classification_report(y_exp_enc, y_pred_exp, target_names=exp_names, digits=3))

cm_exp = confusion_matrix(y_exp_enc, y_pred_exp, normalize='true')
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_exp, xticklabels=exp_names, yticklabels=exp_names,
            annot=True, fmt='.2f', cmap='Greens', vmin=0, vmax=1, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Exp Stage Confusion Matrix — {best_exp}"); plt.tight_layout(); plt.show()

---
# Part D: Summary — All Targets Compared

In [ ]:
print("=" * 72)
print("  SUMMARY: Multi-Target Prediction from 192D hBehaveMAE Embeddings")
print("  Pooling: mean | Chunks: 3000/1800 | CV: GroupShuffleSplit on animal_id")
print("=" * 72)

summary_rows = []

# Strain
best_s = max(strain_cv, key=lambda m: strain_cv[m].mean())
summary_rows.append({"Target": "Strain", "Best Model": best_s,
    "Balanced Acc (grouped)": f"{strain_cv[best_s].mean():.3f} ± {strain_cv[best_s].std():.3f}",
    "# Classes": n_strains, "Chance": f"{1/n_strains:.3f}"})

# Age
best_a = max(age_cv, key=lambda m: age_cv[m]["grouped"].mean())
summary_rows.append({"Target": "Age", "Best Model": best_a,
    "Balanced Acc (grouped)": f"{age_cv[best_a]['grouped'].mean():.3f} ± {age_cv[best_a]['grouped'].std():.3f}",
    "# Classes": n_ages, "Chance": f"{1/n_ages:.3f}"})

# Exp Stage
best_e = max(exp_cv, key=lambda m: exp_cv[m]["grouped"].mean())
summary_rows.append({"Target": "Exp Stage", "Best Model": best_e,
    "Balanced Acc (grouped)": f"{exp_cv[best_e]['grouped'].mean():.3f} ± {exp_cv[best_e]['grouped'].std():.3f}",
    "# Classes": n_exp, "Chance": f"{1/n_exp:.3f}"})

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

## Stage Ablation: All 3 Encoder Stages × All 3 Targets

In [ ]:
print("Stage ablation (LogReg L2, GroupShuffleSplit):\n")
print(f"{'Stage':>8s} {'Dim':>5s}  {'Strain':>12s}  {'Age':>12s}  {'Exp Stage':>12s}")
print("-" * 58)

for stage_name in ["stage1", "stage2", "stage3"]:
    vecs, strains_s, ages_s, exps_s, grps_s = [], [], [], [], []

    with h5py.File(EMB_H5_PATH, 'r') as f:
        for vid in sorted(f.keys()):
            if stage_name not in f[vid]: continue
            emb = f[vid][stage_name][:]
            parts = vid.split('_')
            if len(parts) < 4: continue
            aid, exp_s, age_s = parts[0], parts[1], parts[2]
            strain = strain_lookup.get(aid)
            if strain is None or strain == HYBRID_STRAIN: continue

            for chunk in extract_independent_chunks(emb, CHUNK_FRAMES, GAP_FRAMES):
                vecs.append(chunk.mean(axis=0))
                strains_s.append(strain); ages_s.append(age_s)
                exps_s.append(exp_s); grps_s.append(aid)

    Xs = np.array(vecs)
    mask = np.isin(strains_s, list(valid_strains))
    Xs, ys, ya, ye, gs = (Xs[mask], le_strain.transform(np.array(strains_s)[mask]),
        le_age.transform(np.array(ages_s)[mask]), le_exp.transform(np.array(exps_s)[mask]),
        np.array(grps_s)[mask])

    clf = make_pipeline(StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    gss_s = GroupShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_STATE)

    acc_strain = cross_val_score(clf, Xs, ys, groups=gs, cv=gss_s, scoring="balanced_accuracy").mean()
    acc_age    = cross_val_score(clf, Xs, ya, groups=gs, cv=gss_s, scoring="balanced_accuracy").mean()
    acc_exp    = cross_val_score(clf, Xs, ye, groups=gs, cv=gss_s, scoring="balanced_accuracy").mean()

    print(f"{stage_name:>8s} {Xs.shape[1]:>5d}  {acc_strain:>12.3f}  {acc_age:>12.3f}  {acc_exp:>12.3f}")